In [29]:
!pip install -q pypdf langchain-text-splitters sentence-transformers google-genai

In [30]:
import os
import re
import urllib.request
import numpy as np

from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from google.colab import userdata
from google import genai


# ============================================================
# 1. CONFIGURACIÓN
# ============================================================

URL_PDF = (
    "https://raw.githubusercontent.com/"
    "kimberlyn05/sales-knowledge-ai-agent/"
    "main/data/documents/Politicas_Internas_NexaShip.pdf"
)

RUTA_PDF = "/content/Politicas_Internas_NexaShip.pdf"

MODELO_EMBEDDINGS = (
    "sentence-transformers/"
    "paraphrase-multilingual-MiniLM-L12-v2"
)


# ============================================================
# 2. DESCARGAR PDF
# ============================================================

if not os.path.exists(RUTA_PDF):
    urllib.request.urlretrieve(URL_PDF, RUTA_PDF)
    print("✓ PDF descargado desde GitHub")
else:
    print("✓ PDF disponible")


# ============================================================
# 3. EXTRAER TEXTO POR PÁGINA
# ============================================================

lector = PdfReader(RUTA_PDF)

paginas = []

for numero_pagina, pagina in enumerate(lector.pages, start=1):
    texto = pagina.extract_text() or ""

    paginas.append({
        "pagina": numero_pagina,
        "texto": texto
    })

print(f"✓ Páginas procesadas: {len(paginas)}")


# ============================================================
# 4. LIMPIAR TEXTO
# ============================================================

def limpiar_texto(texto):
    texto = texto.replace("\x7f", " ")
    texto = texto.replace("\u00a0", " ")

    texto = re.sub(r"[ \t]+", " ", texto)
    texto = re.sub(r"\n{3,}", "\n\n", texto)

    return texto.strip()


paginas_limpias = []

for pagina in paginas:
    paginas_limpias.append({
        "pagina": pagina["pagina"],
        "texto": limpiar_texto(pagina["texto"])
    })

print("✓ Texto limpiado")


# ============================================================
# 5. CREAR CHUNKS
# ============================================================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks_limpios = []
chunk_id = 1

for pagina in paginas_limpias:
    fragmentos = text_splitter.split_text(
        pagina["texto"]
    )

    for fragmento in fragmentos:
        chunks_limpios.append({
            "chunk_id": chunk_id,
            "pagina": pagina["pagina"],
            "texto": fragmento
        })

        chunk_id += 1

print(f"✓ Chunks creados: {len(chunks_limpios)}")


# ============================================================
# 6. CARGAR MODELO DE EMBEDDINGS
# ============================================================

modelo_embeddings = SentenceTransformer(
    MODELO_EMBEDDINGS
)

textos_chunks = [
    chunk["texto"]
    for chunk in chunks_limpios
]

embeddings_chunks = modelo_embeddings.encode(
    textos_chunks,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print(
    f"✓ Embeddings generados: "
    f"{len(embeddings_chunks)}"
)


# ============================================================
# 7. CONFIGURAR GEMINI
# ============================================================

gemini_api_key = userdata.get(
    "GEMINI_API_KEY"
)

cliente_gemini = genai.Client(
    api_key=gemini_api_key
)

print("✓ Cliente Gemini configurado")


# ============================================================
# 8. FUNCIÓN DE BÚSQUEDA SEMÁNTICA
# ============================================================

def buscar_evidencia(pregunta, top_k=5):

    embedding_pregunta = modelo_embeddings.encode(
        [pregunta],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    similitudes = (
        embeddings_chunks
        @ embedding_pregunta
    )

    indices_relevantes = np.argsort(
        similitudes
    )[::-1][:top_k]

    resultados = []

    for indice in indices_relevantes:
        chunk = chunks_limpios[indice]

        resultados.append({
            "chunk_id": chunk["chunk_id"],
            "pagina": chunk["pagina"],
            "similitud": float(
                similitudes[indice]
            ),
            "texto": chunk["texto"]
        })

    return resultados


print("✓ Búsqueda semántica preparada")


# ============================================================
# 9. FUNCIÓN DEL AGENTE
# ============================================================

def responder_con_evidencia(
    pregunta,
    top_k=5
):

    evidencias = buscar_evidencia(
        pregunta,
        top_k=top_k
    )

    contexto = "\n\n".join(
        [
            (
                f"[Fuente: página "
                f"{evidencia['pagina']}]\n"
                f"{evidencia['texto']}"
            )
            for evidencia in evidencias
        ]
    )

    prompt = f"""
Eres Sales Knowledge AI Agent,
un asistente interno de NexaShip.

Tu tarea es responder preguntas
utilizando exclusivamente la evidencia
documental proporcionada.

Reglas obligatorias:
- No inventes información.
- No utilices conocimiento externo.
- No completes datos faltantes mediante suposiciones.
- Si la evidencia no permite responder con seguridad,
  indícalo claramente.
- Responde en español.
- Sé claro y profesional.
- Menciona las páginas utilizadas como fuente.
- Si existen varias reglas relevantes,
  intégralas en una sola respuesta coherente.

PREGUNTA:
{pregunta}

EVIDENCIA DOCUMENTAL:
{contexto}

RESPUESTA:
"""

    respuesta = (
        cliente_gemini.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt
        )
    )

    return {
        "pregunta": pregunta,
        "respuesta": respuesta.text,
        "evidencias": evidencias
    }


print("✓ Agente configurado")
print()
print("🚀 Sales Knowledge AI Agent listo")

✓ PDF disponible
✓ Páginas procesadas: 10
✓ Texto limpiado
✓ Chunks creados: 33


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✓ Embeddings generados: 33
✓ Cliente Gemini configurado
✓ Búsqueda semántica preparada
✓ Agente configurado

🚀 Sales Knowledge AI Agent listo


In [31]:
import re
import unicodedata

STOPWORDS = {
    "a", "al", "ante", "como", "con", "cual", "cuando",
    "de", "del", "el", "en", "es", "esta", "este",
    "la", "las", "le", "lo", "los", "me", "mi",
    "no", "ofrece", "para", "por", "puede", "puedo",
    "que", "se", "si", "su", "sus", "un", "una",
    "y", "ya"
}


def normalizar_texto(texto):
    texto = texto.lower()

    texto = unicodedata.normalize(
        "NFD",
        texto
    )

    texto = "".join(
        caracter
        for caracter in texto
        if unicodedata.category(caracter) != "Mn"
    )

    return texto


def extraer_terminos_clave(pregunta):

    pregunta_normalizada = normalizar_texto(
        pregunta
    )

    palabras = re.findall(
        r"\b[a-z0-9]+\b",
        pregunta_normalizada
    )

    return [
        palabra
        for palabra in palabras
        if palabra not in STOPWORDS
        and len(palabra) >= 4
        and palabra != "nexaship"
    ]


def calcular_cobertura_lexica(
    pregunta,
    evidencias
):

    terminos = extraer_terminos_clave(
        pregunta
    )

    texto_evidencias = normalizar_texto(
        " ".join(
            evidencia["texto"]
            for evidencia in evidencias
        )
    )

    encontrados = [
        termino
        for termino in terminos
        if termino in texto_evidencias
    ]

    cobertura = (
        len(encontrados) / len(terminos)
        if terminos
        else 0.0
    )

    return {
        "terminos": terminos,
        "encontrados": encontrados,
        "cobertura": cobertura
    }


print("✓ Validador léxico configurado")

✓ Validador léxico configurado


In [32]:
def clasificar_evidencia(
    pregunta,
    evidencias
):
    if not evidencias:
        return {
            "estado": "insuficiente",
            "mejor_score": 0.0,
            "cobertura_lexica": 0.0
        }

    mejor_score = evidencias[0]["similitud"]

    resultado_lexico = calcular_cobertura_lexica(
        pregunta,
        evidencias
    )

    cobertura_lexica = (
        resultado_lexico["cobertura"]
    )

    if cobertura_lexica == 0.0:
        estado = "insuficiente"

    elif mejor_score >= 0.45:
        estado = "fuerte"

    elif mejor_score >= 0.30:
        estado = "incierta"

    else:
        estado = "insuficiente"

    return {
        "estado": estado,
        "mejor_score": mejor_score,
        "cobertura_lexica": cobertura_lexica
    }


print("✓ Clasificador de evidencia V3 configurado")

✓ Clasificador de evidencia V3 configurado


In [33]:
!pip install -q groq

In [34]:
from google.colab import userdata
from groq import Groq

# Obtener la API key desde los Secrets de Colab
groq_api_key = userdata.get("GROQ_API_KEY")

# Crear cliente de Groq
cliente_groq = Groq(
    api_key=groq_api_key
)

print("✓ Cliente Groq configurado correctamente")

✓ Cliente Groq configurado correctamente


In [35]:
def generar_respuesta_llm(prompt):
    """
    Intenta generar la respuesta con Gemini.
    Si Gemini falla, utiliza Groq como fallback.
    """

    try:
        # Proveedor principal: Gemini
        respuesta_gemini = cliente_gemini.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt
        )

        return {
            "texto": respuesta_gemini.text,
            "proveedor": "gemini",
            "fallback_usado": False
        }

    except Exception as error_gemini:
        print(
            "⚠ Gemini no disponible. "
            "Intentando con Groq..."
        )

        try:
            # Proveedor alternativo: Groq
            respuesta_groq = cliente_groq.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                temperature=0
            )

            return {
                "texto": respuesta_groq.choices[0].message.content,
                "proveedor": "groq",
                "fallback_usado": True
            }

        except Exception as error_groq:
            return {
                "texto": (
                    "No fue posible generar una respuesta "
                    "en este momento porque los proveedores "
                    "de lenguaje no están disponibles."
                ),
                "proveedor": None,
                "fallback_usado": True,
                "error_gemini": str(error_gemini),
                "error_groq": str(error_groq)
            }


print("✓ Fallback Gemini → Groq configurado")

✓ Fallback Gemini → Groq configurado


In [37]:
def responder_agente_controlado(
    pregunta,
    top_k=5
):
    # 1. Recuperar evidencia
    evidencias = buscar_evidencia(
        pregunta,
        top_k=top_k
    )

    # 2. Clasificar evidencia
    clasificacion = clasificar_evidencia(
    pregunta,
    evidencias
)

    estado = clasificacion["estado"]
    mejor_score = clasificacion["mejor_score"]

    # 3. Bloquear consultas claramente sin evidencia
    # Aquí NO se llama a ningún LLM
    if estado == "insuficiente":
        return {
            "pregunta": pregunta,
            "respuesta": (
                "No encontré evidencia documental suficiente "
                "en las fuentes disponibles para responder "
                "esta consulta con seguridad."
            ),
            "estado_evidencia": estado,
            "mejor_score": mejor_score,
            "evidencias": evidencias,
            "llamada_llm": False,
            "proveedor": None,
            "fallback_usado": False
        }

    # 4. Construir contexto documental
    contexto = "\n\n".join(
        [
            (
                f"[Fuente: página "
                f"{evidencia['pagina']}]\n"
                f"{evidencia['texto']}"
            )
            for evidencia in evidencias
        ]
    )

    # 5. Instrucciones según nivel de evidencia
    if estado == "incierta":
        instruccion_evidencia = """
La relevancia de la evidencia recuperada es incierta.

Debes ser especialmente conservador:
- Responde solo si la evidencia contiene información
  directamente aplicable a la pregunta.
- No deduzcas reglas que no estén expresadas.
- No conviertas similitudes temáticas en hechos.
- Si la evidencia no responde directamente,
  indica que no existe información suficiente.
"""
    else:
        instruccion_evidencia = """
La evidencia recuperada presenta una relevancia fuerte.

Aun así:
- Responde exclusivamente con base en la evidencia.
- No agregues información externa.
- No realices suposiciones no respaldadas.
"""

    # 6. Construir prompt
    prompt = f"""
Eres Sales Knowledge AI Agent,
un asistente interno de NexaShip.

Tu tarea es responder preguntas utilizando
exclusivamente la evidencia documental proporcionada.

Reglas obligatorias:
- No inventes información.
- No utilices conocimiento externo.
- No completes datos faltantes mediante suposiciones.
- Si la evidencia no permite responder con seguridad,
  indícalo claramente.
- Responde en español.
- Sé claro y profesional.
- Menciona las páginas utilizadas como fuente.

ESTADO DE EVIDENCIA:
{estado}

INSTRUCCIONES ADICIONALES:
{instruccion_evidencia}

PREGUNTA:
{pregunta}

EVIDENCIA DOCUMENTAL:
{contexto}

RESPUESTA:
"""

    # 7. Generar respuesta con fallback automático
    resultado_llm = generar_respuesta_llm(
        prompt
    )

    # 8. Retornar resultado completo
    return {
        "pregunta": pregunta,
        "respuesta": resultado_llm["texto"],
        "estado_evidencia": estado,
        "mejor_score": mejor_score,
        "evidencias": evidencias,
        "llamada_llm": True,
        "proveedor": resultado_llm["proveedor"],
        "fallback_usado": resultado_llm["fallback_usado"]
    }


print("✓ Agente controlado con fallback configurado")

✓ Agente controlado con fallback configurado


In [38]:
casos_evaluacion = [
    {
        "id": "EVAL-001",
        "pregunta": "¿Puedo prometerle a un cliente una integración personalizada?",
        "categoria": "comercial",
        "esperado_evidencia": True
    },
    {
        "id": "EVAL-002",
        "pregunta": "¿Qué debo hacer si no tengo evidencia suficiente?",
        "categoria": "politica",
        "esperado_evidencia": True
    },
    {
        "id": "EVAL-003",
        "pregunta": "¿Cómo debo escalar una consulta según la skill?",
        "categoria": "escalamiento",
        "esperado_evidencia": True
    },
    {
        "id": "EVAL-004",
        "pregunta": "¿Cuántos días de vacaciones tienen los empleados de NexaShip?",
        "categoria": "fuera_documento",
        "esperado_evidencia": False
    },
    {
        "id": "EVAL-005",
        "pregunta": "¿Cuál es el menú de la cafetería de NexaShip?",
        "categoria": "fuera_documento",
        "esperado_evidencia": False
    }
]

print(f"✓ Dataset de evaluación creado: {len(casos_evaluacion)} casos")

✓ Dataset de evaluación creado: 5 casos


In [39]:
casos_validacion = [
    {
        "id": "VAL-001",
        "pregunta": "¿Qué servicios puede presentar el equipo comercial como disponibles?",
        "esperado_evidencia": True
    },
    {
        "id": "VAL-002",
        "pregunta": "¿Qué debe hacer una persona comercial ante un requerimiento técnico no documentado?",
        "esperado_evidencia": True
    },
    {
        "id": "VAL-003",
        "pregunta": "¿Se puede compartir información interna de aprobaciones con un cliente?",
        "esperado_evidencia": True
    },
    {
        "id": "VAL-004",
        "pregunta": "¿Cuál es el salario promedio de los empleados de NexaShip?",
        "esperado_evidencia": False
    },
    {
        "id": "VAL-005",
        "pregunta": "¿NexaShip ofrece seguro médico a sus empleados?",
        "esperado_evidencia": False
    }
]

print(
    f"✓ Dataset de validación creado: "
    f"{len(casos_validacion)} casos"
)

✓ Dataset de validación creado: 5 casos


In [40]:
preguntas_prueba_v3 = [
    "¿Cuántos días de vacaciones tienen los empleados de NexaShip?",
    "¿NexaShip ofrece seguro médico a sus empleados?"
]

for pregunta in preguntas_prueba_v3:

    resultado = responder_agente_controlado(
        pregunta
    )

    print("PREGUNTA:")
    print(resultado["pregunta"])

    print("\nRESPUESTA:")
    print(resultado["respuesta"])

    print("\nESTADO DE EVIDENCIA:")
    print(resultado["estado_evidencia"])

    print("\nMEJOR SCORE:")
    print(f"{resultado['mejor_score']:.4f}")

    print("\n¿SE LLAMÓ AL LLM?")
    print(resultado["llamada_llm"])

    print("\nPROVEEDOR:")
    print(resultado["proveedor"])

    print("\n¿SE USÓ FALLBACK?")
    print(resultado["fallback_usado"])

    print("\n" + "=" * 100 + "\n")

PREGUNTA:
¿Cuántos días de vacaciones tienen los empleados de NexaShip?

RESPUESTA:
No encontré evidencia documental suficiente en las fuentes disponibles para responder esta consulta con seguridad.

ESTADO DE EVIDENCIA:
insuficiente

MEJOR SCORE:
0.3148

¿SE LLAMÓ AL LLM?
False

PROVEEDOR:
None

¿SE USÓ FALLBACK?
False


PREGUNTA:
¿NexaShip ofrece seguro médico a sus empleados?

RESPUESTA:
No encontré evidencia documental suficiente en las fuentes disponibles para responder esta consulta con seguridad.

ESTADO DE EVIDENCIA:
insuficiente

MEJOR SCORE:
0.4648

¿SE LLAMÓ AL LLM?
False

PROVEEDOR:
None

¿SE USÓ FALLBACK?
False




In [41]:
todos_los_casos = (
    casos_evaluacion
    + casos_validacion
)

aciertos = 0

for caso in todos_los_casos:

    evidencias = buscar_evidencia(
        caso["pregunta"],
        top_k=5
    )

    clasificacion = clasificar_evidencia(
        caso["pregunta"],
        evidencias
    )

    evidencia_detectada = (
        clasificacion["estado"]
        != "insuficiente"
    )

    acierto = (
        evidencia_detectada
        == caso["esperado_evidencia"]
    )

    if acierto:
        aciertos += 1

    print(f"CASO: {caso['id']}")
    print(f"Pregunta: {caso['pregunta']}")
    print(
        f"Esperado evidencia: "
        f"{caso['esperado_evidencia']}"
    )
    print(
        f"Estado: "
        f"{clasificacion['estado']}"
    )
    print(
        f"Score: "
        f"{clasificacion['mejor_score']:.4f}"
    )
    print(
        f"Cobertura léxica: "
        f"{clasificacion['cobertura_lexica']:.4f}"
    )
    print(
        f"Resultado: "
        f"{'✓ ACIERTO' if acierto else '✗ ERROR'}"
    )
    print("-" * 80)


total = len(todos_los_casos)

precision = (
    aciertos / total * 100
)

print()
print("EVALUACIÓN FINAL DEL CLASIFICADOR")
print("=" * 50)
print(f"Total de casos: {total}")
print(f"Aciertos: {aciertos}/{total}")
print(f"Precisión: {precision:.2f}%")

CASO: EVAL-001
Pregunta: ¿Puedo prometerle a un cliente una integración personalizada?
Esperado evidencia: True
Estado: fuerte
Score: 0.6006
Cobertura léxica: 0.7500
Resultado: ✓ ACIERTO
--------------------------------------------------------------------------------
CASO: EVAL-002
Pregunta: ¿Qué debo hacer si no tengo evidencia suficiente?
Esperado evidencia: True
Estado: incierta
Score: 0.3354
Cobertura léxica: 0.2000
Resultado: ✓ ACIERTO
--------------------------------------------------------------------------------
CASO: EVAL-003
Pregunta: ¿Cómo debo escalar una consulta según la skill?
Esperado evidencia: True
Estado: fuerte
Score: 0.6571
Cobertura léxica: 0.8000
Resultado: ✓ ACIERTO
--------------------------------------------------------------------------------
CASO: EVAL-004
Pregunta: ¿Cuántos días de vacaciones tienen los empleados de NexaShip?
Esperado evidencia: False
Estado: insuficiente
Score: 0.3148
Cobertura léxica: 0.0000
Resultado: ✓ ACIERTO
--------------------------